In [6]:
from dotenv import load_dotenv
import os
from huggingface_hub import login
import torch

load_dotenv()
hf_token = os.environ["HUGGINGFACE_HUB_TOKEN"]
login(token=hf_token)
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "meta-llama/Llama-3.2-3B-Instruct"

print(len(hf_token) != 0)
print(device)

True
cuda


In [2]:
from pathlib import Path

with open("data-rag.jsonl","w", encoding="utf-8") as out:
    for path in Path("data").rglob("*"):
        if path.name == "data.jsonl":
            with path.open("r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        out.write(line.rstrip("\n") + "\n")

In [7]:
import json
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)

data = []
with open('data-rag.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

def convert_data(item):
    stripped_prompt = " ".join(item["prompt"].split(" ")[1:])
    conversation =  [{
                "role": "user",
                "content": stripped_prompt
            }, {
                "role": "assistant",
                "content": item["completion"]
            }]
    text = tokenizer.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {
        "text": text
    }


data = [convert_data(item) for item in data]
    
with open('data-rag.jsonl', 'w', encoding='utf-8') as f:
     for item in data:
        f.write(json.dumps(item) + '\n')

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto").to("cuda")

# LoRA config
lora_config = LoraConfig(
    r=4,                      
    lora_alpha=8,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

model = get_peft_model(model, lora_config)

model.tie_weights()
model.print_trainable_parameters()

2026-02-06 20:58:12.777481: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9373] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-06 20:58:12.777528: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-06 20:58:12.779081: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1534] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-06 20:58:12.787552: I tensorflow/core/platform/cpu_feature_guard.cc:183] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Skipping import of cpp extensions due t

trainable params: 6,078,464 || all params: 3,218,828,288 || trainable%: 0.1888


In [3]:
from datasets import load_dataset
dataset = load_dataset("json", data_files="data-rag.jsonl", split="train")

Generating train split: 722 examples [00:00, 142179.80 examples/s]


In [4]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=TrainingArguments(
        output_dir="llama323-finetuned",
        per_device_train_batch_size=4,   # small batch size for Colab GPU
        gradient_accumulation_steps=4,   # accumulate gradients to simulate larger batch
        num_train_epochs=6,
        learning_rate=5e-5,
        logging_steps=50,
        save_strategy="epoch",
        warmup_ratio=0.05,
        weight_decay=0.01,
        lr_scheduler_type="cosine"
    ),
)

trainer.train()
model.save_pretrained("llama323-dnd-finetuned")

Truncating train dataset: 100%|██████████| 722/722 [00:00<00:00, 102186.18 examples/s]
The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.


Step,Training Loss
50,2.848300
100,1.458500
150,1.314000
200,1.245500
250,1.195700


In [7]:
model.push_to_hub("nightfury2986/llama323-dnd-finetuned")
tokenizer.push_to_hub("nightfury2986/llama323-dnd-finetuned")

Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  50%|█████     | 12.3MB / 24.4MB, 61.4MB/s  
Processing Files (0 / 1):  92%|█████████▏| 22.3MB / 24.4MB, 55.8MB/s  
Processing Files (0 / 1):  99%|█████████▊| 24.0MB / 24.4MB, 40.0MB/s  
Processing Files (1 / 1): 100%|██████████| 24.4MB / 24.4MB, 25.0MB/s  
Processing Files (1 / 1): 100%|██████████| 24.4MB / 24.4MB, 24.4MB/s  
New Data Upload: 100%|██████████| 24.4MB / 24.4MB, 24.4MB/s  
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 17.2MB / 17.2MB,  0.00B/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  


CommitInfo(commit_url='https://huggingface.co/nightfury2986/llama323-dnd-finetuned/commit/f991913e1abbb9e53b01e4adf5586d2413935770', commit_message='Upload tokenizer', commit_description='', oid='f991913e1abbb9e53b01e4adf5586d2413935770', pr_url=None, repo_url=RepoUrl('https://huggingface.co/nightfury2986/llama323-dnd-finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='nightfury2986/llama323-dnd-finetuned'), pr_revision=None, pr_num=None)